# Download Models to Local Triton Repository

This notebook downloads all models to a local `triton-repo` folder structure:

```
triton-repo/
├── models/           # Model configs and code
│   ├── yolov8n/
│   │   ├── config.pbtxt
│   │   └── 1/model.onnx
│   ├── bert-base-uncased/
│   │   ├── config.pbtxt
│   │   └── 1/model.onnx
│   ├── whisper-tiny-python/
│   │   ├── config.pbtxt
│   │   ├── packages/humanize/  # pip-installed package
│   │   └── 1/model.py
│   └── smollm-135m-python/
│       ├── config.pbtxt
│       └── 1/model.py
└── weights/          # Model weights (for Python backend models)
    ├── whisper-tiny-python/1/
    └── smollm-135m-python/1/
```

**Run cells in order.** Each model can be downloaded independently.

## 1. Setup - Create Folder Structure

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# =============================================================================
# Configuration - Edit TRITON_REPO_PATH if needed
# =============================================================================
TRITON_REPO_PATH = Path("../triton-repo").resolve()

# Reference models (checked into git with config.pbtxt and model.py)
TRITON_REPO_REFERENCE = Path("../triton-repo-reference").resolve()

# =============================================================================
# Create folder structure
# =============================================================================
models_dir = TRITON_REPO_PATH / "models"
weights_dir = TRITON_REPO_PATH / "weights"

models_dir.mkdir(parents=True, exist_ok=True)
weights_dir.mkdir(parents=True, exist_ok=True)

print(f"Triton repository path: {TRITON_REPO_PATH}")
print(f"  models/ : {models_dir}")
print(f"  weights/: {weights_dir}")
print(f"\nReference models: {TRITON_REPO_REFERENCE}")
print("\nFolder structure created successfully!")

## 2. Download YOLOv8n (Object Detection - ONNX)

Downloads YOLOv8n from Ultralytics and exports to ONNX format.

**Requirements:** `pip install ultralytics onnx onnxruntime`

In [ ]:
MODEL_NAME = "yolov8n"

print(f"{'='*60}")
print(f"Downloading {MODEL_NAME}")
print(f"{'='*60}\n")

# Check dependencies
try:
    from ultralytics import YOLO
    import onnx
    import onnxruntime as ort
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Install with: pip install ultralytics onnx onnxruntime")
    raise

# Create model directory
model_dir = models_dir / MODEL_NAME
version_dir = model_dir / "1"
version_dir.mkdir(parents=True, exist_ok=True)
onnx_path = version_dir / "model.onnx"

# Download and export
print("Loading YOLOv8n from Ultralytics Hub...")
model = YOLO("yolov8n.pt")

print("Exporting to ONNX (opset=17, dynamic batch)...")
export_path = model.export(
    format="onnx",
    imgsz=640,
    opset=17,
    simplify=True,
    dynamic=True,
)

# Move to triton-repo
if Path(export_path).exists():
    shutil.move(export_path, onnx_path)
    print(f"Moved ONNX model to: {onnx_path}")

# Clean up .pt file
if Path("yolov8n.pt").exists():
    Path("yolov8n.pt").unlink()

# Verify
print("\nVerifying ONNX model...")
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print(f"  Inputs: {[(i.name, i.shape) for i in session.get_inputs()]}")
print(f"  Outputs: {[(o.name, o.shape) for o in session.get_outputs()]}")

# Create config.pbtxt
config_content = '''name: "yolov8n"
platform: "onnxruntime_onnx"
max_batch_size: 16

input [
  {
    name: "images"
    data_type: TYPE_FP32
    dims: [ 3, 640, 640 ]
  }
]

output [
  {
    name: "output0"
    data_type: TYPE_FP32
    dims: [ 84, -1 ]
  }
]

instance_group [
  {
    kind: KIND_CPU
    count: 1
  }
]

dynamic_batching {
  preferred_batch_size: [ 1, 2, 4, 8 ]
  max_queue_delay_microseconds: 100
}
'''

config_path = model_dir / "config.pbtxt"
with open(config_path, "w") as f:
    f.write(config_content)

print(f"\n✓ {MODEL_NAME} downloaded successfully!")
print(f"  Model: {onnx_path}")
print(f"  Config: {config_path}")

## 3. Download BERT (Text Classification - ONNX)

Downloads BERT from HuggingFace and exports to ONNX format.

**Requirements:** `pip install transformers torch onnx onnxruntime`

In [ ]:
MODEL_NAME = "bert-base-uncased"

print(f"{'='*60}")
print(f"Downloading {MODEL_NAME}")
print(f"{'='*60}\n")

# Check dependencies
try:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    import onnx
    import onnxruntime as ort
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Install with: pip install transformers torch onnx onnxruntime")
    raise

# Create model directory
model_dir = models_dir / MODEL_NAME
version_dir = model_dir / "1"
version_dir.mkdir(parents=True, exist_ok=True)
onnx_path = version_dir / "model.onnx"

# Download model and tokenizer
print(f"Downloading {MODEL_NAME} from HuggingFace...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model.eval()

# Create dummy inputs
max_seq_length = 128
dummy_text = "This is a sample text for model export."
inputs = tokenizer(
    dummy_text,
    padding="max_length",
    truncation=True,
    max_length=max_seq_length,
    return_tensors="pt",
)

# Export to ONNX
print("Exporting to ONNX (opset=14)...")
with torch.no_grad():
    torch.onnx.export(
        model,
        (inputs["input_ids"], inputs["attention_mask"]),
        str(onnx_path),
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "logits": {0: "batch_size"},
        },
        opset_version=14,
        do_constant_folding=True,
    )

# Save tokenizer
print("Saving tokenizer...")
tokenizer.save_pretrained(str(version_dir))

# Verify
print("\nVerifying ONNX model...")
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print(f"  Inputs: {[(i.name, i.shape) for i in session.get_inputs()]}")
print(f"  Outputs: {[(o.name, o.shape) for o in session.get_outputs()]}")

# Create config.pbtxt
config_content = '''name: "bert-base-uncased"
platform: "onnxruntime_onnx"
max_batch_size: 0

input [
  {
    name: "input_ids"
    data_type: TYPE_INT64
    dims: [ -1, -1 ]
  },
  {
    name: "attention_mask"
    data_type: TYPE_INT64
    dims: [ -1, -1 ]
  }
]

output [
  {
    name: "logits"
    data_type: TYPE_FP32
    dims: [ -1, 2 ]
  }
]

instance_group [
  {
    kind: KIND_CPU
    count: 1
  }
]
'''

config_path = model_dir / "config.pbtxt"
with open(config_path, "w") as f:
    f.write(config_content)

print(f"\n✓ {MODEL_NAME} downloaded successfully!")
print(f"  Model: {onnx_path}")
print(f"  Tokenizer: {version_dir}")
print(f"  Config: {config_path}")

## 4. Download Whisper (Audio Transcription - Python Backend)

Downloads Whisper from HuggingFace and sets up Python backend.

**Requirements:** `pip install transformers torch "optimum[onnxruntime]" librosa soundfile humanize`

In [ ]:
MODEL_NAME = "whisper-tiny-python"
MODEL_ID = "openai/whisper-tiny"

print(f"{'='*60}")
print(f"Downloading {MODEL_NAME}")
print(f"{'='*60}\n")

# Check dependencies
try:
    import torch
    from transformers import WhisperProcessor
    from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
    import humanize
except ImportError as e:
    print(f"Missing dependency: {e}")
    print('Install with: pip install transformers torch "optimum[onnxruntime]" librosa soundfile humanize')
    raise

# Create directories
model_dir = models_dir / MODEL_NAME
version_dir = model_dir / "1"
packages_dir = model_dir / "packages"
weights_path = weights_dir / MODEL_NAME / "1"

version_dir.mkdir(parents=True, exist_ok=True)
packages_dir.mkdir(parents=True, exist_ok=True)
weights_path.mkdir(parents=True, exist_ok=True)

# Download weights
print(f"Downloading model weights to: {weights_path}")
if not (weights_path / "config.json").exists():
    print("Downloading processor...")
    processor = WhisperProcessor.from_pretrained(MODEL_ID)
    
    print("Downloading model and exporting to ONNX (this may take a while)...")
    model = ORTModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, export=True)
    
    print("Saving weights...")
    processor.save_pretrained(str(weights_path))
    model.save_pretrained(str(weights_path))
else:
    print("Weights already exist, skipping download.")

# Install humanize package to packages directory
print(f"\nInstalling humanize package to: {packages_dir}")
import subprocess
subprocess.run([
    sys.executable, "-m", "pip", "install", 
    "--target", str(packages_dir), 
    "--upgrade", "--quiet",
    "humanize"
], check=True)
print(f"  Installed humanize to {packages_dir}")

# Copy model.py from reference
source_model_py = TRITON_REPO_REFERENCE / "models" / MODEL_NAME / "1" / "model.py"
target_model_py = version_dir / "model.py"

if source_model_py.exists():
    shutil.copy2(source_model_py, target_model_py)
    print(f"Copied model.py from reference")
else:
    print(f"WARNING: model.py not found at {source_model_py}")

# Create config.pbtxt
config_content = '''name: "whisper-tiny-python"
backend: "python"
max_batch_size: 8

input [
  {
    name: "input_features"
    data_type: TYPE_FP32
    dims: [ 80, 3000 ]
  },
  {
    name: "language"
    data_type: TYPE_STRING
    dims: [ 1 ]
    optional: true
  },
  {
    name: "task"
    data_type: TYPE_STRING
    dims: [ 1 ]
    optional: true
  }
]

output [
  {
    name: "transcription"
    data_type: TYPE_STRING
    dims: [ 1 ]
  },
  {
    name: "token_ids"
    data_type: TYPE_INT64
    dims: [ 448 ]
  }
]

instance_group [
  {
    kind: KIND_CPU
    count: 1
  }
]

parameters {
  key: "model_type"
  value: { string_value: "audio" }
}

parameters {
  key: "model_id"
  value: { string_value: "openai/whisper-tiny" }
}

parameters {
  key: "use_onnx"
  value: { string_value: "true" }
}

dynamic_batching {
  preferred_batch_size: [ 1, 2, 4 ]
  max_queue_delay_microseconds: 100000
}
'''

config_path = model_dir / "config.pbtxt"
with open(config_path, "w") as f:
    f.write(config_content)

print(f"\n✓ {MODEL_NAME} downloaded successfully!")
print(f"  Model: {version_dir}")
print(f"  Weights: {weights_path}")
print(f"  Packages: {packages_dir}")
print(f"  Config: {config_path}")

## 5. Download SmolLM-135M (Text Generation - Python Backend)

Downloads SmolLM-135M-Instruct from HuggingFace.

**Requirements:** `pip install transformers torch`

In [ ]:
MODEL_NAME = "smollm-135m-python"
MODEL_ID = "HuggingFaceTB/SmolLM-135M-Instruct"

print(f"{'='*60}")
print(f"Downloading {MODEL_NAME}")
print(f"{'='*60}\n")

# Check dependencies
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from huggingface_hub import hf_hub_download
    import json
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Install with: pip install transformers torch huggingface_hub")
    raise

# Create directories
model_dir = models_dir / MODEL_NAME
version_dir = model_dir / "1"
weights_path = weights_dir / MODEL_NAME / "1"

version_dir.mkdir(parents=True, exist_ok=True)
weights_path.mkdir(parents=True, exist_ok=True)

# Download weights
print(f"Downloading model weights to: {weights_path}")

# Download tokenizer files
print("Downloading tokenizer files...")
tokenizer_files = [
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "special_tokens_map.json",
]
for filename in tokenizer_files:
    try:
        cached_path = hf_hub_download(MODEL_ID, filename)
        dest_path = weights_path / filename
        shutil.copy(cached_path, dest_path)
        print(f"  Downloaded {filename}")
    except Exception as e:
        print(f"  Skipping {filename}: {e}")

# Fix tokenizer config for compatibility
tokenizer_config_path = weights_path / "tokenizer_config.json"
if tokenizer_config_path.exists():
    with open(tokenizer_config_path, "r") as f:
        config = json.load(f)
    if config.get("tokenizer_class") != "GPT2Tokenizer":
        config["tokenizer_class"] = "GPT2Tokenizer"
        with open(tokenizer_config_path, "w") as f:
            json.dump(config, f, indent=2)
        print("  Set tokenizer_class to GPT2Tokenizer for compatibility")

# Download model weights
print("Downloading model weights...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    low_cpu_mem_usage=True,
)
model.save_pretrained(weights_path)
print(f"  Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

# Copy model.py from reference
source_model_py = TRITON_REPO_REFERENCE / "models" / MODEL_NAME / "1" / "model.py"
target_model_py = version_dir / "model.py"

if source_model_py.exists():
    shutil.copy2(source_model_py, target_model_py)
    print(f"Copied model.py from reference")
else:
    print(f"WARNING: model.py not found at {source_model_py}")

# Create config.pbtxt
config_content = '''name: "smollm-135m-python"
backend: "python"
max_batch_size: 0

input [
  {
    name: "prompt"
    data_type: TYPE_STRING
    dims: [ 1 ]
  },
  {
    name: "max_tokens"
    data_type: TYPE_INT32
    dims: [ 1 ]
    optional: true
  },
  {
    name: "temperature"
    data_type: TYPE_FP32
    dims: [ 1 ]
    optional: true
  },
  {
    name: "top_p"
    data_type: TYPE_FP32
    dims: [ 1 ]
    optional: true
  }
]

output [
  {
    name: "generated_text"
    data_type: TYPE_STRING
    dims: [ 1 ]
  },
  {
    name: "token_count"
    data_type: TYPE_INT32
    dims: [ 1 ]
  }
]

instance_group [
  {
    kind: KIND_CPU
    count: 1
  }
]

parameters {
  key: "model_type"
  value: { string_value: "text-llm" }
}

parameters {
  key: "model_id"
  value: { string_value: "HuggingFaceTB/SmolLM-135M-Instruct" }
}
'''

config_path = model_dir / "config.pbtxt"
with open(config_path, "w") as f:
    f.write(config_content)

print(f"\n✓ {MODEL_NAME} downloaded successfully!")
print(f"  Model: {version_dir}")
print(f"  Weights: {weights_path}")
print(f"  Config: {config_path}")

## 6. Summary - List Downloaded Models

In [ ]:
print(f"{'='*60}")
print("Downloaded Models Summary")
print(f"{'='*60}\n")

print(f"Triton Repository: {TRITON_REPO_PATH}\n")

# List models
print("Models:")
for model_path in sorted(models_dir.iterdir()):
    if model_path.is_dir():
        config_exists = (model_path / "config.pbtxt").exists()
        version_dir = model_path / "1"
        has_onnx = (version_dir / "model.onnx").exists()
        has_model_py = (version_dir / "model.py").exists()
        has_packages = (model_path / "packages").exists()
        
        model_type = "ONNX" if has_onnx else "Python" if has_model_py else "Unknown"
        packages_info = " [+packages]" if has_packages else ""
        config_info = "✓" if config_exists else "✗"
        
        print(f"  [{config_info}] {model_path.name} ({model_type}){packages_info}")

# List weights
print("\nWeights:")
if weights_dir.exists():
    for weights_path in sorted(weights_dir.iterdir()):
        if weights_path.is_dir():
            size = sum(f.stat().st_size for f in weights_path.rglob("*") if f.is_file())
            size_str = f"{size / 1024 / 1024:.1f} MB"
            print(f"  {weights_path.name} ({size_str})")
else:
    print("  (none)")

print(f"\nReady for deployment!")

---
## 7. Copy Models to EDV and Run Tests (Optional)

This cell copies models to the EDV (External Data Volume) and runs the sequential test script.

**Prerequisites:**
- Set environment variables for your deployment
- Ensure EDV is mounted at `TARGET_TRITON_ROOT`

In [ ]:
# =============================================================================
# Configuration for EDV copy and testing
# =============================================================================
import os

# Set these for your environment
NAMESPACE = "domino-inference-dev"

os.environ["TRITON_REST_URL"] = f"http://triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:8080"
os.environ["TRITON_GRPC_URL"] = f"triton-inference-server-proxy.{NAMESPACE}.svc.cluster.local:50051"
os.environ["SOURCE_TRITON_ROOT"] = str(TRITON_REPO_PATH)
os.environ["TARGET_TRITON_ROOT"] = f"/domino/edv/{NAMESPACE}-triton-repo-pvc"

print("Environment configured:")
print(f"  TRITON_REST_URL: {os.environ['TRITON_REST_URL']}")
print(f"  TRITON_GRPC_URL: {os.environ['TRITON_GRPC_URL']}")
print(f"  SOURCE_TRITON_ROOT: {os.environ['SOURCE_TRITON_ROOT']}")
print(f"  TARGET_TRITON_ROOT: {os.environ['TARGET_TRITON_ROOT']}")

In [ ]:
# =============================================================================
# Run sequential model tests (with copy to EDV)
# =============================================================================
import subprocess

# Models to test (comment out any you don't want to test)
MODELS_TO_TEST = [
    "bert-base-uncased",
    "yolov8n",
    "whisper-tiny-python",
    "smollm-135m-python",
]

# Build command
cmd = [
    sys.executable,
    "../scripts/testing/test_models_sequential.py",
    "--models", *MODELS_TO_TEST,
    # "--skip-copy",      # Uncomment to skip copying to EDV
    # "--skip-benchmark", # Uncomment to skip benchmarks
    # "--cpu",            # Uncomment to use KIND_CPU instead of KIND_GPU
]

print(f"Running: {' '.join(cmd)}\n")
print("="*60)

# Run the test script
result = subprocess.run(cmd, cwd=str(Path.cwd()))

if result.returncode == 0:
    print("\n" + "="*60)
    print("✓ All tests passed!")
else:
    print("\n" + "="*60)
    print(f"✗ Tests failed with exit code: {result.returncode}")